# 🤖 Agentic Skill-to-Career Mapper
This notebook demonstrates an AI agent that researches career trends and finds real-time job openings using LangChain and Google Gemini.

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -qU langchain-google-genai

In [ ]:
!pip install -q langchain-tavily

In [ ]:
!pip install -q langchain-tavily

> ### **Step 1: Library Imports & Model Initialization**
> *We start by importing the necessary LangChain components and initializing the Google Gemini model using secure API keys.*

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from google.colab import userdata
from langchain_tavily import TavilySearch

In [ ]:
import requests
from langchain.tools import tool
from pprint import pprint

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
gemini_api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
    'google_genai:gemini-2.5-flash',
    api_key=gemini_api_key
)

In [ ]:
tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_search_tool = TavilySearch(
    max_results=10,
    tavily_api_key=tavily_api_key,
    search_depth = "advanced"
)

> ### **Step 2: Custom Tool Definition**
> *To give the agent 'real-world' capabilities, we define a custom tool that interacts with the JSearch API to fetch live job listings based on skills and location.*

In [ ]:
rapid_api_key= userdata.get('RAPID_API_KEY')

In [ ]:
@tool
def search_jobs(skill: str, location: str) -> list:
    """
    Search for jobs based on a skill and location.
    """
    pprint("Calling search job tools")
    pprint(f"Skill: {skill}")
    pprint(f"Location: {location}")
    url = "https://jsearch.p.rapidapi.com/search"

    querystring = {
        "query":f"{skill} jobs in {location}",
        "page":"1",
        "num_pages":"1",
        "country":"in",
        "employment_types":"INTERN,FULLTIME",
        "job_requirements":"no_experience,under_3_years_experience",
        }

    headers = {
    	"x-rapidapi-key": rapid_api_key,
    	"x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)

    data = response.json()
    jobs = data['data']

    result = []
    for job in jobs:
        result.append({
            "company_name": job.get('employer_name',""),
            "job_title": job.get('job_title',""),
            "job_description": job.get('job_description',""),
            "job_link": job.get('job_apply_link',"")
        })

    return result

> ### **Step 3: Agent Configuration**
> *Here, we define the system prompt (the agent's personality) and bind the tools to the LLM. We also implement InMemorySaver to enable conversation memory.*

In [ ]:
### Note: Re-importing and initializing in one block for clean execution
!pip install -q langchain-tavily
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from google.colab import userdata
from langchain_tavily import TavilySearch
import requests
from langchain.tools import tool
from pprint import pprint

# Initialization of API keys and models/tools
gemini_api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
    'google_genai:gemini-2.5-flash',
    api_key=gemini_api_key
)

tavily_api_key = userdata.get('TAVILY_API_KEY')
skill_demand_search_tool = TavilySearch(
    max_results=10,
    tavily_api_key=tavily_api_key,
    search_depth = "advanced"
)

rapid_api_key = userdata.get('RAPID_API_KEY')

@tool
def search_jobs(skill: str, location: str) -> list:
    """
    Search for jobs based on a skill and location.
    """
    pprint("Calling search job tools")
    pprint(f"Skill: {skill}")
    pprint(f"Location: {location}")
    url = "https://jsearch.p.rapidapi.com/search"

    querystring = {
        "query":f"{skill} jobs in {location}",
        "page":"1",
        "num_pages":"1",
        "country":"in",
        "employment_types":"INTERN,FULLTIME",
        "job_requirements":"no_experience,under_3_years_experience",
        }

    headers = {
    	"x-rapidapi-key": rapid_api_key,
    	"x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)

    data = response.json()
    jobs = data['data']

    result = []
    for job in jobs:
        result.append({
            "company_name": job.get('employer_name',""),
            "job_title": job.get('job_title',""),
            "job_description": job.get('job_description',""),
            "job_link": job.get('job_apply_link',"")
        })

    return result

system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.\n\nYou have access to these tools:\n- skill_demand_tool: Search for industry demand, salary insights, and career trends\n- search_jobs: Find actual job listings requiring specific skills\n\nHelp the student by researching the skill they ask about and finding relevant opportunities.\n\nPresent results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

checkpointer = InMemorySaver()

agent = create_agent(
    model = model,
    tools = [skill_demand_search_tool, search_jobs],
    system_prompt = system_prompt,
    checkpointer = checkpointer
)

In [ ]:
config = {
    "configurable":{
        "thread_id":"chat-1"
    }
}

> ### **Step 4: Execution & Querying**
> *Finally, we invoke the agent with a user query. The agent will decide whether to research trends or search for jobs based on the input.*

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
user_query = "Which jobs are most suitable for candidates who are familiar with Web Application Development?"
response = agent.invoke({
    "messages":[{"role":"user" , "content":user_query}]
},config=config)

pprint(response["messages"][-1].content)

[{'extras': {'signature': 'CtkKAQw51scyS51MnAsZWd52hG++U/PDmjLdc7NZACAcnW/8kEJh/9y3R8IHsMKtIwaGo0oYiU2WDXkR3kQDQQPXQRpkICHHw18x305MEo7s/BCoD6HDOus83RXGMMx0m4pk4VcSspd2sWLd5RVPBFHBO6QH2HvvlBhLsNjgjJfmU+etZA1StpJtqGogUEmba17kXYe5/VAe+8C6hVSTU+DIDOuJ5xRANikRMGBQNzqWdVNYr4K/qGsWe44ZGxWIs6EMXlZ4TVl3OPQBSohDnlpfEwgM94y0vavut4TP+7rbgGjfziqcOCVo74ONRYz+8xgVYFE9/jX+Cb4v7ITtE4MnJ7q/PL9gBqm/yJi4cGWPgbMA5wmY1FpS6xBY2BtkPHxmn9MMV6CI5H7LJPADD4vVpXhtXuSLjw87y3cbMEHsqDAC/lp89r1+iP5WLFH37ZkYLl79JiwBZwExZEnOfZ7dvfalbsXIWu9WTVsxfZvdVvj2qdkfzAGsxxAygaMRlMdZBtVXO24dEwqBYMO1GjCmgMFKWuW74aWUUWlHU2+WrZ3dmqXKAOTLV3T12QchPa45Kd/IN4GhM6V/sgR1R/1s4Lb+0OXk6nvkwyov+65fFs/MeeCydxlAcrHLq3WCYFZj4NeTwKoTGNVCefZbwZXO8sKQeDxeGlbRcFjRy7jlPTuQPMvbEQY8g0yw5oNKIJcWZbD3a8VQbjqWrWyQjnC2Udb85ajaC06gllBRgA1zifxdeh9eQS4CQiqpThmthvgMtx4l+HtAHg3ja/1L/KFtPyqHeEWxE1txzKvJrgGx9u1/YHUzjTqojjBabhq94Ou1W4c+5Wauv+ZBJyr2cyiPGP/n2aWvlkCvzrfRmOOv+VSz947rKPm2moWHGVngjTeimCEiloPC52ud2enmKtVc40ZBqCmpOPIBRzfMQdTizGQYQsf01WfJb8asBpLYTsOw2nMwCN9q1

### 🚀 Project Overview: AI-Powered Skill-to-Career Mapper

This project demonstrates the implementation of an **Agentic AI System** using the **LangGraph** and **LangChain** frameworks. The goal is to bridge the gap between skill acquisition and job market opportunities.

#### 🛠️ Key Technical Components:

1.  **Orchestration Layer (LangGraph & LangChain):**
    *   Utilizes `create_agent` to build a conversational agent capable of reasoning and executing tasks.
    *   **Memory Management:** Implements `InMemorySaver` to provide the agent with 'short-term memory' (thread-based state), allowing for context-aware multi-turn conversations.

2.  **Model Integration:**
    *   Powered by **Google Gemini 2.5 Flash** (via `langchain-google-genai`). This model provides high-speed reasoning and complex instruction following.

3.  **Real-Time Tooling (Function Calling):**
    *   **Market Intelligence:** Integrated **Tavily Search** for deep-web research on industry trends, salary insights, and skill demand.
    *   **Live Job Retrieval:** Developed a custom tool using the **JSearch API** (RapidAPI) to fetch real-time, entry-level job listings filtered by specific skill sets and locations.

4.  **Security & Best Practices:**
    *   Employs Google Colab's `userdata` secrets management to handle sensitive API keys (Gemini, Tavily, RapidAPI) securely.

#### 📈 Professional Value Proposition:
This application isn't just a chatbot; it's a **Decision Support System**. It automates the research phase of career planning by fetching live data from disparate sources, synthesizing it, and providing actionable job opportunities based on real-time market conditions.

### 📚 External Resources & API Credits

To replicate or extend this project, the following external services and APIs were utilized:

1.  **Google Gemini API:** Provides the core LLM capabilities for reasoning and natural language generation.
    *   *Source:* [Google AI Studio](https://aistudio.google.com/)

2.  **Tavily Search API:** A search engine optimized for LLMs and AI agents, used here for real-time market research and trend analysis.
    *   *Source:* [Tavily AI](https://tavily.com/)

3.  **JSearch API (via RapidAPI):** Used to retrieve real-time job listings from across the web. It provides structured data for job titles, descriptions, and application links.
    *   *Source:* [JSearch on RapidAPI](https://rapidapi.com/letscrape-6bRBaP_failure/api/jsearch)

4.  **LangGraph & LangChain:** The framework used for building the agentic workflow and managing conversation state.

# 🤖 Agentic Skill-to-Career Mapper

This project demonstrates an AI agent built using LangChain and LangGraph that helps users research career trends and find real-time job openings. It acts as a Decision Support System, automating the research phase of career planning by fetching live data from disparate sources, synthesizing it, and providing actionable job opportunities based on real-time market conditions.

## 🚀 Project Overview

This project implements an **Agentic AI System** to bridge the gap between skill acquisition and job market opportunities.

### 🛠️ Key Technical Components:

1.  **Orchestration Layer (LangGraph & LangChain):**
    *   Utilizes `create_agent` to build a conversational agent capable of reasoning and executing tasks.
    *   **Memory Management:** Implements `InMemorySaver` to provide the agent with 'short-term memory' (thread-based state), allowing for context-aware multi-turn conversations.

2.  **Model Integration:**
    *   Powered by **Google Gemini 2.5 Flash** (via `langchain-google-genai`). This model provides high-speed reasoning and complex instruction following.

3.  **Real-Time Tooling (Function Calling):**
    *   **Market Intelligence:** Integrated **Tavily Search** for deep-web research on industry trends, salary insights, and skill demand.
    *   **Live Job Retrieval:** Developed a custom tool using the **JSearch API** (RapidAPI) to fetch real-time, entry-level job listings filtered by specific skill sets and locations.

4.  **Security & Best Practices:**
    *   Employs Google Colab's `userdata` secrets management to handle sensitive API keys (Gemini, Tavily, RapidAPI) securely.

## 📚 External Resources & API Credits

To replicate or extend this project, the following external services and APIs are utilized:

1.  **Google Gemini API:** Provides the core LLM capabilities for reasoning and natural language generation.
    *   *Source:* [Google AI Studio](https://aistudio.google.com/)

2.  **Tavily Search API:** A search engine optimized for LLMs and AI agents, used here for real-time market research and trend analysis.
    *   *Source:* [Tavily AI](https://tavily.com/)

3.  **JSearch API (via RapidAPI):** Used to retrieve real-time job listings from across the web. It provides structured data for job titles, descriptions, and application links.
    *   *Source:* [JSearch on RapidAPI](https://rapidapi.com/letscrape-6bRBaP_failure/api/jsearch)

## ⚙️ Installation

To run this notebook, you'll need to install the necessary Python packages. You can do this by running the following commands in your notebook:

```bash
!pip install -q langchain
!pip install -qU langchain-google-genai
!pip install -q langchain-tavily
```

## 🔑 API Key Setup

This project requires API keys for Google Gemini, Tavily Search, and JSearch (via RapidAPI). Please follow these steps to set them up securely:

1.  **Obtain API Keys:**
    *   **Google Gemini:** Get your API key from [Google AI Studio](https://aistudio.google.com/).
    *   **Tavily Search:** Obtain your API key from [Tavily AI](https://tavily.com/).
    *   **JSearch (RapidAPI):** Subscribe to the JSearch API on [RapidAPI](https://rapidapi.com/letscrape-6bRBaP_failure/api/jsearch) to get your `x-rapidapi-key`.

2.  **Store in Google Colab Secrets:**
    *   In Google Colab, go to the left panel, click on the "🔑 Secrets" icon.
    *   Add your API keys with the following names:
        *   `GEMINI_API_KEY`
        *   `TAVILY_API_KEY`
        *   `RAPID_API_KEY`

## 🚀 Usage

Once the libraries are installed and API keys are configured, you can run the notebook cells sequentially. The agent is configured to respond to user queries about skill demand and job opportunities.

### Example Query:

```python
user_query = "Which jobs are most suitable for candidates who are familiar with Web Application Development?"
response = agent.invoke({
    "messages":[{"role":"user" , "content":user_query}]
},config=config)

pprint(response["messages"][-1].content)
```

The agent will then use the `skill_demand_search_tool` to research the skill and the `search_jobs` tool to find relevant job listings, presenting the results in a structured format.